In [ ]:
import json
import time
from pathlib import Path
import requests

# 1. Dizin Yapılandırması (Directory Configuration)
# Doğru klasöre gitmesi için
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Kesin hedef: proje-koku/data/raw
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print(f"Verilerin kaydedileceği hedef dizin: {raw_dir.resolve()}")

# Dizi Listesi ve API Parametreleri
# 12 ikonik yapımın TVMaze ID eşlemeleri
SHOWS = {
    "The Simpsons": 83,
    "Doctor Who": 210,
    "Friends": 431,
    "Breaking Bad": 169,
    "Game of Thrones": 82,
    "The Office": 526,
    "Brooklyn Nine-Nine": 49,
    "Grey's Anatomy": 67,
    "Supernatural": 71,
    "Stranger Things": 2993,
    "Black Mirror": 305,
    "Sherlock": 335,
}

raw_data = []
print("TVMaze API üzerinden veri toplama süreci başlatıldı...\n")

# Veri Çekme Döngüsü (Data Ingestion)
for name, show_id in SHOWS.items():
    # episodes endpoint'i diziye ait tüm bölüm detaylarını tek seferde döner
    url = f"https://api.tvmaze.com/shows/{show_id}?embed=episodes"
    
    try:
        # Zaman aşımı ve ağ hatası kontrolleri
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        show_json = response.json()
        raw_data.append(show_json)
        
        episodes_count = len(show_json.get("_embedded", {}).get("episodes", []))
        print(f"✓ {name:<18} -> {episodes_count} bölüm başarıyla çekildi.")
        
    except requests.exceptions.HTTPError as http_err:
        print(f"✗ HTTP Hatası ({name}): {http_err}")
    except requests.exceptions.RequestException as err:
        print(f"✗ Bağlantı Hatası ({name}): {err}")
        
    # API rate-limit sınırı
    time.sleep(0.3)

# Ham JSON Dosyasının Kaydedilmesi
output_file = raw_dir / "tvmaze_raw.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(raw_data, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print(f"İşlem tamamlandı! Toplam çekilen dizi: {len(raw_data)}")
print(f"Ham JSON dosyası başarıyla kaydedildi:\n-> {output_file.resolve()}")
print("=" * 60)

Verilerin kaydedileceği hedef dizin: C:\Users\HUAWEI\OneDrive\Masaüstü\API'dan veriye- Marvel Dataseti\data\raw
TVMaze API üzerinden veri toplama süreci başlatıldı...

✓ The Simpsons       -> 808 bölüm başarıyla çekildi.
✓ Doctor Who         -> 153 bölüm başarıyla çekildi.
✓ Friends            -> 236 bölüm başarıyla çekildi.
✓ Breaking Bad       -> 62 bölüm başarıyla çekildi.
✓ Game of Thrones    -> 73 bölüm başarıyla çekildi.
✓ The Office         -> 202 bölüm başarıyla çekildi.
✓ Brooklyn Nine-Nine -> 153 bölüm başarıyla çekildi.
✓ Grey's Anatomy     -> 471 bölüm başarıyla çekildi.
✓ Supernatural       -> 519 bölüm başarıyla çekildi.
✓ Stranger Things    -> 42 bölüm başarıyla çekildi.
✓ Black Mirror       -> 32 bölüm başarıyla çekildi.
✓ Sherlock           -> 12 bölüm başarıyla çekildi.

İşlem tamamlandı! Toplam çekilen dizi: 12
Ham JSON dosyası başarıyla kaydedildi:
-> C:\Users\HUAWEI\OneDrive\Masaüstü\API'dan veriye- Marvel Dataseti\data\raw\tvmaze_raw.json


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd


# 1. Proje Kök Dizinini (Root) Belirleme
# Notebook notebooks/ içindeyse bir üst dizine (proje köküne) çıkar
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

raw_file = project_root / "data" / "raw" / "tvmaze_raw.json"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 2. Ham Veriyi data/raw Altından Yükleme
with open(raw_file, "r", encoding="utf-8") as f:
    records = json.load(f)

# 3. pd.json_normalize ile Üst Bilgileri Düzleştirme (Flattening)
shows_df = pd.json_normalize(records)

cols_to_keep = [
    "id", "name", "type", "language", "genres", 
    "runtime", "averageRuntime", "rating.average", 
    "network.name", "schedule.days", "_embedded.episodes"
]
shows_df = shows_df[[c for c in cols_to_keep if c in shows_df.columns]]

# 4. explode ile Bölümleri Satırlara Açma ve Birleştirme
# Bölüm listesini satırlara patlatma
df_exploded = shows_df.explode("_embedded.episodes").reset_index(drop=True)

# İç içe bölüm objesini normalize etme
episodes_df = pd.json_normalize(df_exploded["_embedded.episodes"])
episodes_df = episodes_df.add_prefix("episode_")

# Dizi ve bölüm tablolarını yatayda birleştirme
final_df = pd.concat([df_exploded.drop(columns=["_embedded.episodes"]), episodes_df], axis=1)

# Sütun isimlerini sadeleştirme
rename_dict = {
    "name": "show_name",
    "rating.average": "show_avg_rating",
    "network.name": "network",
    "episode_id": "episode_id",
    "episode_name": "episode_name",
    "episode_season": "season",
    "episode_number": "episode_num",
    "episode_runtime": "episode_runtime",
    "episode_rating.average": "episode_rating",
    "episode_airdate": "airdate"
}
final_df = final_df.rename(columns=rename_dict)


# 5. Veri Temizleme Adımları
# a. Tekrar eden kayıtların kaldırılması
final_df = final_df.drop_duplicates(subset=["episode_id"])

# b. Veri tipleri dönüşümü
final_df["season"] = pd.to_numeric(final_df["season"], errors="coerce")
final_df["episode_num"] = pd.to_numeric(final_df["episode_num"], errors="coerce")
final_df["episode_rating"] = pd.to_numeric(final_df["episode_rating"], errors="coerce")
final_df["episode_runtime"] = pd.to_numeric(final_df["episode_runtime"], errors="coerce")
final_df["airdate"] = pd.to_datetime(final_df["airdate"], errors="coerce")

# c. Yayın günü ve birincil tür çıkarma
final_df["air_day"] = final_df["schedule.days"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)
final_df["primary_genre"] = final_df["genres"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unspecified"
)

# d. Eksik puan kontrolü: Puanı olmayan bölümleri filtreleme
clean_df = final_df.dropna(subset=["episode_rating"]).copy()

# 6. Temizlenmiş Veriyi data/processed Klasörüne CSV Olarak Yazma
clean_csv_path = processed_dir / "tvmaze_episodes_clean.csv"
clean_df.to_csv(clean_csv_path, index=False)

print("✓ İşlem başarıyla tamamlandı!")
print(f"✓ Temizlenmiş veri kaydedildi: {clean_csv_path.resolve()}")
print(f"✓ Analize hazır toplam bölüm sayısı: {len(clean_df)}")

✓ İşlem başarıyla tamamlandı!
✓ Temizlenmiş veri kaydedildi: C:\Users\HUAWEI\OneDrive\Masaüstü\API'dan veriye- Kendi Datasetim\data\processed\tvmaze_episodes_clean.csv
✓ Analize hazır toplam bölüm sayısı: 2302
